In [ ]:
import pandas as pd
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

def setup_chrome_driver():
    """Setup Chrome driver in headless mode"""
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    return webdriver.Chrome(options=chrome_options)

def html_to_image(driver, html_content, output_path):
    """Convert HTML content to image using Selenium with full page capture"""
    full_html = f"""
    <html>
        <head>
            <style>
                body {{
                    margin: 0;
                    padding: 0;
                    background: white;
                }}
                .question-wrapper {{
                    padding: 0px;
                    background: white;
                    display: inline-block;
                    max-width: none;
                    margin-bottom: 50px;  /* Add bottom margin for safety */
                }}
                img {{
                    height: auto;
                    max-width: 100%;
                    display: block;
                }}
            </style>
        </head>
        <body>
            <div class="question-wrapper">
                {html_content}
            </div>
            <script>
                // Force recalculation of heights
                document.addEventListener('DOMContentLoaded', function() {{
                    window.dispatchEvent(new Event('resize'));
                }});
            </script>
        </body>
    </html>
    """

    temp_html = output_path.with_suffix('.html')
    with open(temp_html, 'w', encoding='utf-8') as f:
        f.write(full_html)

    driver.get(f'file://{temp_html.absolute()}')

    # Increase wait time for content to load
    time.sleep(3)

    # Get the required height by getting the content height plus padding
    required_height = driver.execute_script("""
        return Math.ceil(document.querySelector('.question-wrapper').getBoundingClientRect().height) + 100;
    """)

    # Set the window size with extra padding
    driver.set_window_size(800, required_height + 100)

    # Wait for resize to take effect
    time.sleep(1)

    # Take full page screenshot
    element = driver.find_element('css selector', '.question-wrapper')

    # Scroll to ensure element is in view and wait
    driver.execute_script("arguments[0].scrollIntoView(true);", element)
    time.sleep(1)

    # Take screenshot
    element.screenshot(str(output_path))

    # Cleanup
    temp_html.unlink()

def extract_question_html(csv_path, output_dir):
    """Extract question content and save as images"""
    # Create output directory if it doesn't exist
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Setup Chrome driver
    driver = setup_chrome_driver()

    try:
        # Read CSV file
        df = pd.read_csv(csv_path)

        # Forward fill item_id to handle empty values
        df['item_id'] = df['item_id'].ffill()

        # Group by item_id
        grouped = df.groupby('item_id', dropna=False)

        for item_id, group in grouped:
            # Get all item descriptions for this group
            descriptions = [desc for desc in group['item_description'] if pd.notna(desc) and desc.strip()]
            description = descriptions[0] if descriptions else ""

            # Forward fill question_type to handle empty values
            group['question_type'] = group['question_type'].ffill()

            # Initialize lists to store contents for each question type
            contents = []
            options_html = ""

            # Process each unique question type in the group
            for q_type in group['question_type'].unique():
                if pd.notna(q_type):
                    # Get content for this question type
                    type_group = group[group['question_type'] == q_type]
                    type_contents = [content for content in type_group['question_content'] if pd.notna(content)]
                    contents.extend(type_contents)

                    # Handle MCQ options for this type
                    if q_type == 'MCQ':
                        options = type_group['options'].dropna().tolist()
                        if options:
                            options_html += "<div style='margin-top: 15px;'><h4>Options:</h4>"
                            for opt in options:
                                options_html += f"<div style='margin-left: 25px; margin-bottom: 10px;'>{opt}</div>"
                            options_html += "</div>"

            # Only create entry if there's content to show
            if description or contents:
                combined_html = f"""
                <div style="border: 1px solid #ccc; padding: 20px; margin: 10px;">
                    {description}
                    {"".join([f'<div style="margin-top: 15px;">{content}</div>' for content in contents])}
                    {options_html}
                </div>
                """

                # Create safe filename from item_id
                # safe_filename = "".join(c if c.isalnum() else "_" for c in str(item_id))
                output_path = output_dir / f"{item_id}.png"

                # Convert HTML to image
                html_to_image(driver, combined_html, output_path)
                print(f"Saved image for item_id: {item_id}")

    finally:
        driver.quit()

# Main execution
if __name__ == "__main__":
    csv_path = Path("/home/lawtrann/Workspaces/math_tutor/src/math_tutor/tools/notebook/question_content.csv")
    output_dir = Path("/home/lawtrann/Workspaces/math_tutor/src/math_tutor/tools/notebook/question_images")

    # Extract questions and save as images
    extract_question_html(csv_path, output_dir)